# 02 · Fourier Optics & Single-Frame Phase Retrieval

**Sistem Fotonika — Digital Fringe Processing**  
**Iwan Cony Setiadi · Teknik Fisika**  
**Level:** S1 semester 6 dengan pengayaan riset · **Mode:** teori + implementasi + eksperimen numerik

> **Status data:** semua citra dan angka dalam notebook ini berasal dari **model/simulasi sintetis** dengan *ground truth*, bukan pengukuran instrumen nyata. Asumsi dan keterbatasan dibahas dalam setiap eksperimen.

**Tujuan:** memahami domain frekuensi spasial dan mengimplementasikan metode Takeda berbasis FFT tanpa fungsi black box.

## Tujuan, konteks, dan model


**Pertanyaan riset:** kapan satu citra fringe sudah cukup untuk mengekstraksi fase? Jawabannya **tidak selalu**: Fourier single-frame memerlukan *spatial carrier* yang memisahkan sideband dari DC dan spektrum konjugat. Kita akan menguji kegagalan ketika frekuensi carrier terlalu kecil, noise besar, atau pemfilteran menghilangkan detail fase.

Model: $$I(x,y)=A(x,y)+B(x,y)\cos[2\pi f_c x+\varphi(x,y)]+\eta(x,y).$$

$x,y$ adalah indeks piksel, $f_c$ dalam siklus/piksel, $A,B$ intensitas relatif, dan $\varphi$ fase objek (rad). Transformasi Fourier 2D menggunakan pasangan $(f_x,f_y)$ dengan satuan siklus/piksel; proses perkalian oleh $\exp(i2\pi f_c x)$ memindahkan spektrum objek ke sideband $+f_c$. Semua contoh memakai model intensitas skalar sintetis dan carrier sejajar sumbu $x$.

## 1 · Derivasi matematis metode Takeda


Dengan identitas Euler:

$$I=A+\frac{B}{2}e^{i\varphi}e^{i2\pi f_cx}+\frac{B}{2}e^{-i\varphi}e^{-i2\pi f_cx}.$$

Pada domain Fourier, komponen $A$ berada di sekitar frekuensi nol, sedangkan informasi fase muncul dalam dua **sideband konjugat**. Isolasi salah satu sideband lalu inverse FFT memberikan medan kompleks (termodulasi):

$$C(x,y)\simeq \frac{B(x,y)}2 e^{i[2\pi f_cx+\varphi(x,y)]}.$$

Setelah demodulasi carrier $C_d=C\,e^{-i2\pi f_cx}$, berlaku $\varphi_w=\arg C_d\in(-\pi,\pi]$. Ketika objek halus, $\varphi$ dapat dipulihkan melalui *unwrapping* dan penentuan konstanta fase (referensi).

**Batas:** pemisahan spektral yang tidak memadai menimbulkan *bias*. Mask Fourier terlalu sempit membuang detail (*oversmoothing*); terlalu lebar memasukkan DC/noise/sideband lain. Filtering bersifat aproksimasi, bukan inversi unik dari satu citra umum. Ini adalah implementasi edukasional dari keluarga pendekatan Takeda, bukan klaim replikasi perangkat tertentu.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage
from scipy import stats

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({'figure.dpi': 105, 'axes.grid': False,
                     'image.interpolation': 'nearest'})
print('NumPy:', np.__version__)

In [ ]:
from skimage.restoration import unwrap_phase

ny,nx=192,256
y,x=np.mgrid[:ny,:nx]
fc=32/nx # 32 siklus melintasi lebar citra; tepat di bin FFT
phi_true=(0.90*np.exp(-((x-125)/54)**2-((y-94)/47)**2)
          +0.23*np.sin(2*np.pi*y/ny))
A=0.53+0.025*np.cos(2*np.pi*y/ny)
B=0.36*(0.96+0.04*np.cos(2*np.pi*x/nx))
I=A+B*np.cos(2*np.pi*fc*x+phi_true)
print('Carrier:',fc,'siklus/px; bin FFT:',int(fc*nx))
fig,ax=plt.subplots(1,2,figsize=(10.5,3.6),constrained_layout=True)
ax[0].imshow(I,cmap='gray');ax[0].set_title('Citra fringe sintetis')
im=ax[1].imshow(phi_true,cmap='twilight');ax[1].set_title('Ground-truth fase objek [rad]');plt.colorbar(im,ax=ax[1]);plt.show()

## 1b · Fourier optics fisik: bidang fokus lensa dan sistem 4f

Sebelum FFT digunakan sebagai *software*, transformasi Fourier sudah muncul secara **fisik** pada bidang fokus belakang lensa tipis dalam rezim difraksi Fraunhofer (dengan syarat iluminasi koheren, kondisi paraxial, dan faktor fase yang sesuai). Untuk medan kompleks $U_0(x,y)$ di bidang masukan, medan pada bidang fokus $(x_f,y_f)$ secara proporsional:

$$U_f(x_f,y_f)\propto\mathcal{F}\{U_0(x,y)\}\bigg|_{f_x=x_f/(\lambda f),\;f_y=y_f/(\lambda f)}.$$

- $f$ panjang fokus lensa [m], $\lambda$ panjang gelombang [m], $f_x,f_y$ frekuensi spasial [siklus/m], dan $x_f,y_f$ koordinat fisik bidang Fourier [m].
- Sistem **4f** dengan dua lensa menempatkan *spatial filter* pada bidang Fourier di antara lensa. Filter yang membuang orde difraksi bisa menekan informasi fringe, bukan meningkatkan detail secara ajaib.
- FFT numerik pada notebook berikutnya mengoperasikan citra **intensitas** hasil kamera; itu berbeda dari transformasi optik koheren atas **amplitudo kompleks** $U_0$. Kita harus tahu besaran apa yang ditransformasikan.

Contoh berikut menghitung medan Fourier secara numerik untuk kisi amplitudo sinusoidal; ini bukan hasil pengukuran lensa fisik. Perkiraan jarak orde difraksi: $x_{f,1}\approx \lambda f/p$, di mana $p$ pitch kisi dalam meter.

In [ ]:
# Simulasi Fourier optics fisik: medan masuk -> bidang fokus lensa -> filter 4f
M=256
pitch_sensor_m=8e-6
lambda_m=532e-9
f_lens_m=0.20
yy,xx=np.mgrid[:M,:M]
xx0=xx-(M-1)/2; yy0=yy-(M-1)/2
pitch_grating_px=16
U0=np.exp(-((xx0/60)**2+(yy0/60)**2)/2)*(1+0.45*np.cos(2*np.pi*xx0/pitch_grating_px))
U_f=np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(U0)))
frequencies=np.fft.fftshift(np.fft.fftfreq(M,d=pitch_sensor_m))
focal_mm=lambda_m*f_lens_m*frequencies*1e3
pitch_grating_m=pitch_grating_px*pitch_sensor_m
first_order_mm=lambda_m*f_lens_m/pitch_grating_m*1e3
# Sistem 4f idealisasi: masker low-pass di bidang Fourier
fy_mm,fx_mm=np.meshgrid(focal_mm,focal_mm,indexing='ij')
pupil=(fx_mm**2+fy_mm**2)<0.52**2
U_filtered=np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(U_f*pupil)))
print(f'Prediksi posisi orde difraksi pertama: ±{first_order_mm:.3f} mm')
assert first_order_mm>0.52 # orde pertama teredam oleh pupil ini
fig,axs=plt.subplots(1,3,figsize=(13.5,3.5),constrained_layout=True)
axs[0].imshow(np.abs(U0)**2,cmap='gray');axs[0].set_title('Intensitas bidang input')
axs[1].imshow(np.log1p(np.abs(U_f)),cmap='magma',extent=[focal_mm[0],focal_mm[-1],focal_mm[-1],focal_mm[0]])
axs[1].set(title='Bidang Fourier lensa',xlabel='$x_f$ [mm]',ylabel='$y_f$ [mm]')
axs[1].set_xlim(-1.8,1.8);axs[1].set_ylim(1.8,-1.8)
axs[2].imshow(np.abs(U_filtered)**2,cmap='gray');axs[2].set_title('Setelah filter spasial ideal 4f')
plt.show()

**Interpretasi:** jarak orde ditentukan oleh $\lambda f/p$, sehingga memilih ukuran aperture bidang Fourier tanpa konversi frekuensi ↔ koordinat fisik dapat merusak desain optik. *Coherent amplitude transfer* melalui pupil juga tidak sama dengan *incoherent OTF/MTF* yang mendeskripsikan pencitraan intensitas pada iluminasi inkomheren. Dalam studi kasus fringe nanti, yang diolah ialah citra intensitas sehingga sideband yang difilter bukan otomatis identik dengan orde difraksi pada eksperimen 4f.

## 2 · FFT 2D dan frekuensi spasial


FFT diskret menganggap data periodik dalam bidang citra. Frekuensi sampel bukan radian/piksel, melainkan siklus/piksel pada `np.fft.fftfreq`; perkalian $2\pi$ mengonversi ke radian/piksel. Untuk carrier 32 siklus/256 piksel, dua sideband muncul di $k_x=\pm32$ bin. Citra riil memiliki simetri Hermitian; mengambil dua sideband sekaligus akan menghapus informasi fase kompleks yang kita inginkan.

In [ ]:
F=np.fft.fftshift(np.fft.fft2(I))
cy,cx=ny//2,nx//2
sidebin=int(round(fc*nx))
fig,ax=plt.subplots(figsize=(9,4))
ax.imshow(np.log10(np.abs(F)+1),cmap='magma')
ax.scatter([cx,cx-sidebin,cx+sidebin],[cy]*3,marker='o',facecolors='none',edgecolors='cyan',s=120)
ax.set(xlabel='Bin $k_x$',ylabel='Bin $k_y$',title='Spektrum Fourier 2D (log10 amplitudo)')
plt.show()

## 3 · Implementasi sideband filter dan demodulasi


Fungsi berikut adalah implementasi *dari prinsip dasar*. Mask radial Gaussian merupakan pilihan filter ilustratif; parameter `sigma_bins` adalah lebar mask dalam **bin FFT**, bukan hertz. Filter ditaruh pada **sideband positif**; carrier dihilangkan setelah inverse FFT. Sinyal hasil tetap memuat bias jika DC dan sideband berdekatan.

Satu hal yang harus diperiksa: tanda fase! Memilih sideband negatif menghasilkan tanda fase berlawanan. Kriteria pemilihan tanda didasarkan pada definisi koordinat, bukan kecocokan visual.

In [ ]:
def takeda_phase(image, carrier_bin_x, sigma_bins=12):
    """Ekstraksi fase off-axis satu citra; carrier di sumbu x dan bin integer."""
    ny,nx=image.shape
    ft=np.fft.fftshift(np.fft.fft2(image))
    ky,kx=np.mgrid[:ny,:nx]
    center_y,center_x=ny//2,nx//2+int(carrier_bin_x)
    mask=np.exp(-0.5*((kx-center_x)**2+(ky-center_y)**2)/sigma_bins**2)
    analytic=np.fft.ifft2(np.fft.ifftshift(ft*mask))
    carrier=np.exp(-2j*np.pi*(carrier_bin_x/nx)*np.arange(nx)[None,:])
    demodulated=analytic*carrier
    return np.angle(demodulated),np.abs(demodulated),mask

def offset_aligned_rmse(estimate,truth,roi):
    delta=np.median((estimate-truth)[roi]) # fase absolut tidak teramati
    return np.sqrt(np.mean((estimate[roi]-truth[roi]-delta)**2))

wrapped,amplitude,fourier_mask=takeda_phase(I,sidebin,sigma_bins=12)
unwrapped=np.unwrap(np.unwrap(wrapped,axis=1),axis=0)
roi=np.zeros_like(I,dtype=bool);roi[26:-26,26:-26]=True
phase_offset=np.median((unwrapped-phi_true)[roi])
phi_est=unwrapped-phase_offset
rmse=np.sqrt(np.mean((phi_est[roi]-phi_true[roi])**2))
print(f'RMSE fase di ROI tengah: {rmse:.4f} rad')
assert rmse < 0.18, 'Periksa posisi sideband dan demodulasi; RMSE terlalu besar.'
fig,ax=plt.subplots(1,4,figsize=(15,3.3),constrained_layout=True)
for a,arr,title in zip(ax,[fourier_mask,wrapped,phi_est,phi_est-phi_true],
  ['Gaussian sideband mask','Wrapped phase [rad]','Unwrapped [rad]','Error [rad]']):
    h=a.imshow(arr,cmap='twilight' if title!='Gaussian sideband mask' else 'viridis')
    a.set_title(title);plt.colorbar(h,ax=a)
plt.show()

## 4 · Eksperimen bandwidth–noise trade-off


Dalam penilaian algoritma penelitian, **akurasi tidak cukup diuji pada satu gambar**. Kita ukur pengaruh lebar mask, noise kamera, dan *region of interest* (ROI) yang sama. Seluruh uji berikut mengukur RMSE fase **setelah menghilangkan konstanta fase global**, karena satu frame tidak memberi referensi fase absolut. Nilai RMSE bukan ukuran ketidakpastian kalibrasi fisik.

In [ ]:
from scipy.ndimage import gaussian_filter
noise_levels=[0.0,0.015,0.035]
widths=[5,9,13,17,23]
results=[]
for sigma_noise in noise_levels:
    rng=np.random.default_rng(12345) # seed sama supaya perbandingan antarfilter fair
    noisy=I+rng.normal(0,sigma_noise,size=I.shape)
    for width in widths:
        phase,_,_=takeda_phase(noisy,sidebin,sigma_bins=width)
        phase=np.unwrap(np.unwrap(phase,axis=1),axis=0)
        error=offset_aligned_rmse(phase,phi_true,roi)
        results.append((sigma_noise,width,error))
fig,ax=plt.subplots(figsize=(8.5,4))
for level in noise_levels:
    d=[r for r in results if r[0]==level]
    ax.plot([r[1] for r in d],[r[2] for r in d],'-o',label=f'σ noise={level}')
ax.set(xlabel='Lebar mask Gaussian [bin FFT]',ylabel='RMSE fase ROI [rad]',
       title='Bias–variance trade-off pada filtering');ax.legend();plt.show()
print('RMSE minimum per noise:')
for level in noise_levels:
    print(level, min([v for n,w,v in results if n==level]))

## 5 · Kasus kegagalan carrier terlalu rendah


Saat $f_c$ terlalu kecil, sideband tumpang tindih dengan DC dan sideband konjugat. Kita **tidak boleh** menyembunyikan kegagalan dengan memilih filter sempit sekali lalu mengklaim resolusi tinggi. Gunakan ground truth untuk membandingkan $f_c=8/N_x$ dan $32/N_x$. Pemilihan frekuensi carrier adalah problem desain optik-sensor, bukan sekadar parameter perangkat lunak.

In [ ]:
errors_carrier=[]
for k in [7,12,20,32,47]:
    signal=A+B*np.cos(2*np.pi*k*x/nx+phi_true)
    wp,_,_=takeda_phase(signal,k,sigma_bins=12)
    up=np.unwrap(np.unwrap(wp,axis=1),axis=0)
    errors_carrier.append(offset_aligned_rmse(up,phi_true,roi))
fig,ax=plt.subplots(figsize=(7.5,3.4))
ax.plot([7,12,20,32,47],errors_carrier,'o-')
ax.set(xlabel='Carrier [bin FFT]',ylabel='RMSE fase ROI [rad]',
       title='Carrier kecil: sideband/DC overlap');plt.show()
assert errors_carrier[0]>errors_carrier[3]

## 6 · Tantangan dan pembahasan riset


**Latihan terstruktur:** (1) ganti Gaussian filter dengan jendela Hann 2D di sekitar sideband; (2) bandingkan RMSE dan kemampuan mempertahankan detail medan fase; (3) uji carrier berarah miring (vektor $\mathbf f_c=(f_x,f_y)$) dan tulis demodulasi 2D yang benar; (4) uji keberhasilan ketika terdapat saturasi dan modulasi rendah.

**Kriteria laporan:** definisi ROI; parameter FFT dan filter; citra asli, spektrum, fase wrapped, fase hasil, residual; RMSE di satuan rad; serta interpretasi kapan metode gagal. **Jangan klaim akurasi metrologi tanpa model kalibrasi dari notebook 04.**

**Literatur primer:** Takeda, Ina, & Kobayashi (1982), JOSA 72:156–160. https://doi.org/10.1364/JOSA.72.000156. Referensi ini diverifikasi pada laman penerbit; teks penuh mungkin memerlukan akses institusional.

<!-- AUDIT-RESEARCH-2026 -->

## 7 · Generalisasi dua dimensi: *oblique carrier*, leakage, dan pilihan ROI

Model Takeda di atas sengaja membatasi carrier pada $x$ dan posisi integer-bin. Agar layak dipakai menganalisis citra yang diputar, perlu generalisasi $\mathbf f_c=(f_{cx},f_{cy})$ dan demodulasi $\exp[-2\pi i(f_{cx}x+f_{cy}y)]$. Ini bukan kosmetik: memfilter $k_x$ saja ketika carrier miring akan melewatkan puncak sideband.

Sinyal nyata dan finite image menghasilkan *spectral leakage* ketika frekuensi carrier bukan kelipatan resolusi DFT. Windowing menurunkan sidelobe namun memperlebar lobe utama, sehingga berpotensi mencampur DC/sideband dan mengurangi resolusi. Semua nilai dalam uji berikut ditentukan sebelum run, bukan dioptimasi pada ground truth eksperimen.

**Desain uji:** carrier 2D $(22/N_x,15/N_y)$; baseline gagal berupa filter horizontal yang keliru; filter 2D memilih kedua koordinat sideband. RMSE dihitung setelah penyelarasan konstanta fase hanya pada ROI tengah yang dinyatakan. Untuk pencitraan nyata, carrier dapat diestimasi dari spektrum menggunakan puncak subpixel dan perlu model ketidakpastiannya.

In [ ]:
# Generalisasi yang mempertahankan konvensi tanda dan menguji filter 2D pada carrier miring
def takeda_phase_2d(img, carrier_bins_yx, sigma_bins=9.):
    ny0,nx0 = img.shape
    ky,kx = np.mgrid[:ny0,:nx0]
    fcb_y,fcb_x = carrier_bins_yx
    mask = np.exp(-.5*((kx-(nx0//2+fcb_x))**2+(ky-(ny0//2+fcb_y))**2)/sigma_bins**2)
    analytic = np.fft.ifft2(np.fft.ifftshift(np.fft.fftshift(np.fft.fft2(img))*mask))
    yg,xg = np.mgrid[:ny0,:nx0]
    demod = analytic*np.exp(-2j*np.pi*(fcb_x*xg/nx0 + fcb_y*yg/ny0))
    return np.angle(demod),mask

oblique_bins=(15,22)
I_oblique = A + B*np.cos(2*np.pi*(oblique_bins[1]*x/nx+oblique_bins[0]*y/ny)+phi_true)
wrapped_2d, mask_oblique = takeda_phase_2d(I_oblique,oblique_bins,9)
phase_2d = unwrap_phase(wrapped_2d)
rmse_2d = offset_aligned_rmse(phase_2d,phi_true,roi)
# Negative control: force wrong sideband y-position (not a competing optimized algorithm)
wrapped_wrong,_ = takeda_phase_2d(I_oblique,(0,oblique_bins[1]),9)
phase_wrong = unwrap_phase(wrapped_wrong)
rmse_wrong = offset_aligned_rmse(phase_wrong,phi_true,roi)
print(f'Oblique 2D RMSE={rmse_2d:.4f} rad; wrong horizontal-only filter={rmse_wrong:.4f} rad')
assert rmse_2d < 0.18 and rmse_wrong > rmse_2d
# Leakage demonstration on a finite 1D signal. Spectral energy outside 2 nearest carrier bins.
t=np.arange(nx)
leak = {}
for kb in [22.,22.35]:
    sig=np.exp(2j*np.pi*kb*t/nx)
    power=np.abs(np.fft.fft(sig))**2
    bin_nearest=int(np.rint(kb))%nx
    leak[kb] = float(1-power[bin_nearest]/power.sum())
print('Complex-tone leakage outside nearest bin:',leak)
assert leak[22.35] > leak[22.] + 0.1
fig,ax=plt.subplots(1,3,figsize=(13,3.5),constrained_layout=True)
ax[0].imshow(I_oblique,cmap='gray');ax[0].set_title('Fringe oblique sintetis')
ax[1].imshow(mask_oblique,cmap='viridis');ax[1].set_title('Mask sideband 2D')
im=ax[2].imshow(phase_2d-np.median((phase_2d-phi_true)[roi])-phi_true,cmap='coolwarm',vmin=-.2,vmax=.2)
ax[2].set_title('Residual phase [rad]');plt.colorbar(im,ax=ax[2]);plt.show()